To understand GitLab's internals, it helps to follow a single action from the browser down to the disk. Let’s look at what happens when you click that **"Merge"** button on a Merge Request (MR).

### The "Life of a Merge" Request Flow

1. **The User Action (UI → NGINX)**
* You click "Merge" in your browser.
* The request hits **NGINX** (the front door). NGINX sees it is a web request and passes it to **Workhorse**.


2. **The Smart Proxy (Workhorse → Puma)**
* **Workhorse** realizes this is a "small" instruction (just a command to merge, not a giant file upload), so it hands it off to **Puma**.
* **Puma** (the Ruby "brain") checks the database (**PostgreSQL**) to make sure you actually have permission to merge this code.


3. **The Background Handoff (Puma → Redis → Sidekiq)**
* Merging can take time, so Puma doesn't do it directly. Instead, it creates a "Job" and drops it into **Redis**.
* **Sidekiq** is constantly watching Redis. It sees the new "Merge Job," picks it up, and starts executing the Ruby code to perform the merge.


4. **The Git Operation (Sidekiq → Gitaly)**
* Sidekiq cannot touch the code files itself. It sends a "Remote Procedure Call" (gRPC) to **Gitaly**.
* **Gitaly** performs the actual `git merge` command on the physical `.git` files stored on the server's disk.


5. **The Wrap Up (Gitaly → Sidekiq → UI)**
* Once Gitaly finishes, it tells Sidekiq, "The files are merged."
* Sidekiq updates the status in **PostgreSQL** from "Open" to "Merged."
* The next time your browser polls the server, **Puma** sees the "Merged" status in the database and shows you the purple "Merged" badge on your screen.



---

### Summary of Responsibilities

| Component | Responsibility |
| --- | --- |
| **Workhorse** | Offloads heavy lifting (large files/long polls) from the main app. |
| **Puma** | Processes the logic and handles your web session. |
| **Sidekiq** | Does the "heavy lifting" in the background so the UI stays fast. |
| **Gitaly** | The only service that touches your Git repositories. |
| **PostgreSQL** | Stores the "metadata" (Who, What, When). |
| **Redis** | The "waiting room" for background tasks. |

**Pro-tip:** If you click Merge and it stays "Merging..." forever, the first place to check is **Sidekiq** (to see if the job is stuck) or **Gitaly** (to see if the disk is slow).

Would you like to know how to view the logs for one of these specific components to troubleshoot a stuck merge?